In this script we are running an optmisation process for both muscles - TA and VL - in where there VthCa deactivation threshold is optmised to match experimental hysteresis

In [1]:
using Plots, Distributions, JLD, DSP, Parameters, Match, SymDict, NumericalIntegration, Random, MAT, Pkg, CSV, DataFrames, XLSX, Pkg, JLD2, Colors, JLD2,  Makie, CairoMakie, Optim, Serialization, JuMP, Ipopt, ForwardDiff, LinearAlgebra, DifferentialEquations, LineSearches, BlackBoxOptim, Random

folder_path = "./functions_optim/"
for file in readdir(folder_path)
    if endswith(file, ".jl")
        include(joinpath(folder_path, file))
    end
end

┌ Warning: Error requiring `MakieCore` from `Polynomials`
│   exception = (LoadError("/Users/nmt121/.julia/packages/Polynomials/6i39P/ext/PolynomialsMakieCoreExt.jl", 1, ArgumentError("Package Polynomials does not have MakieCore in its dependencies:\n- If you have Polynomials checked out for development and have\n  added MakieCore as a dependency but haven't updated your primary\n  environment's manifest file, try `Pkg.resolve()`.\n- Otherwise you may need to report an issue with Polynomials")), Union{Ptr{Nothing}, Base.InterpreterIP}[Ptr{Nothing} @0x000000010d2dd4ae, Ptr{Nothing} @0x000000010b6549bf, Ptr{Nothing} @0x000000010b685a0c, Ptr{Nothing} @0x000000010b684e62, Ptr{Nothing} @0x000000010b684a65, Ptr{Nothing} @0x000000010b685104, Ptr{Nothing} @0x000000010b685c34, Ptr{Nothing} @0x0000000120827d67, Ptr{Nothing} @0x000000010b6549bf, Ptr{Nothing} @0x000000012071828a, Ptr{Nothing} @0x0000000120717d62, Ptr{Nothing} @0x000000010b6549bf, Ptr{Nothing} @0x000000010b661259, Ptr{Nothing} @0x0

## Functions for the creation of the model excluding faciliation effect 

In [ ]:
# df_hyst = CSV.read("/Users/nmt121/Desktop/Repositories/Motoneuron_Model/Data_MN_Model/hysteresis.csv",DataFrame; header=false);
# df_hyst = CSV.read("/Users/nmt121/Desktop/Repositories/Motoneuron_Model/Data_MN_Model/hysteresisVL.csv",DataFrame; header=false);
# hysteresis_exp = df_hyst[:, 1];

hysteresis_exp = CSV.read("/Users/nmt121/Desktop/Repositories/Motoneuron_Model/Data_MN_Model/csv_subjects/hysteresis8VL.csv", DataFrame; header=false)
hysteresis_exp = hysteresis_exp[:, 1]
Plots.scatter(hysteresis_exp, ylim=(-1,0.5))

## Initiliase parameters for running simulation

In [3]:
nr_s = length(hysteresis_exp)
 
# Radius and soma lengths
rheobase_S = LinRange(3.5,6.5,nr_s) 
rs_S = LinRange(38.75,41.25,nr_s)./10000 
ls_S = LinRange(77.5,82.5,nr_s)./10000 
rms_S = LinRange(1.15, 1.05,nr_s).*1000
rd_S = LinRange(20.75,31.25,nr_s)./10000 
ld_S = LinRange(5500.,6800.,nr_s)./10000 
rmd_S = LinRange(14.4,10.70,nr_s).*1000 

#  Maximun conductances for each of the ION channels mS/cm^2 
max_gna = 30.0 
max_gkf_S = ones(nr_s)*4 
max_gks_S =LinRange(16,25,nr_s) 
max_gca_S = LinRange(0.038,0.029,nr_s) 


# Rate constants of the kinetic processes in ms^-1
αm = ones(nr_s)*22.0
βm = ones(nr_s)*13.0

αh_S = ones(nr_s)*0.5
βh_S = ones(nr_s)*4

αn_S = ones(nr_s)*1.5
βn_S = ones(nr_s)*0.1

αq_S = ones(nr_s)*1.5
βq_S = LinRange(0.025,0.038,nr_s) 

αp_D = 0.008
βp_D = LinRange(0.014,0.016,nr_s) 


Apeak_S = LinRange(0.103,0.123,nr_s)
tpeak_S = LinRange(110.,100.,nr_s) 
tetanic_S = LinRange(0.393,0.491,nr_s);


#EXCITATORY PROFILE
amplitude=0.4
tr_length=28000
half_triang = div(tr_length, 2)  
flat_line = fill(-0.1, 1000)
up = range(-0.1, amplitude, length = half_triang + 1)  
down = range(amplitude, -0.1, length = tr_length - half_triang + 1)  
tr = vcat(flat_line, up[2:end], down[2:end], flat_line)
sim_length=30000

# at some point delete these things
γa=1.;γh = ones(nr_s)*1#γn_new
IF= LinRange(1.,1.,nr_s);  #factor for the inhibtion

#reversal potentials
EL=0; ENa=120; ECa=140; EK=-10; Ee=70; Ei=-16 

#maximun conductances
Ge=0.6; Gi=0.5 ; 

#initial voltages
Vd0 = 0.; 
Vs0 = 0.;

#refractory period
rp=5.0; 

const step = 0.05;
_int(num) = floor(Int, num);

#Pulse duration [mS]
pd=1; 

ri = 70.  # Ω.cm 3intracellular resistivity
cm = 1.; #membrane capacitance

Vthca_S= zeros(nr_s); #to initialize

MN_list=[];
Vth_S = zeros(nr_s); gls_S = zeros(nr_s); gld_S = zeros(nr_s); gc_S = zeros(nr_s); gna_S = zeros(nr_s); gkf_S = zeros(nr_s); gks_S = zeros(nr_s); gca_S = zeros(nr_s); rn_S = zeros(nr_s); cs_S = zeros(nr_s)  ; cd_S = zeros(nr_s)
for i in 1:nr_s 
    ##assuming that the somatic membrane can be modeled as a cylinder and using the defined parameters from the previous section
    gls_S[i] = ((2*π*rs_S[i]*ls_S[i])/rms_S[i])*1e6  # conductance of the soma for the ith neuron (uS)
    gld_S[i] = ((2*π*rd_S[i]*ld_S[i])/rmd_S[i])*1e6  # # conductance of the dendrite for the ith neuron (uS)
    gc_S[i] = ((2/(((ri*ld_S[i])/(π*rd_S[i]^2))+((ri*ls_S[i])/(π*rs_S[i]^2)))))*1e6    # coupling conductance in uS
    rn_S[i] = 1/(gls_S[i]+((gld_S[i]*gc_S[i])/(gld_S[i]+gc_S[i]))) # input resistance of the neuron in MΩ     
    Vth_S[i] = round(rn_S[i]*rheobase_S[i],digits=2)   # in mV
    cs_S[i] = 2*π*ls_S[i]*rs_S[i]*cm*1e3   #inner capacitance of the soma nF
    cd_S[i] = 2*π*ld_S[i]*rd_S[i]*cm*1e3   #nF
    
    #conductances uS:
    gna_S[i] = 2*π*ls_S[i]*rs_S[i]*max_gna*1e3          
    gkf_S[i] = 2*π*ls_S[i]*rs_S[i]*max_gkf_S[i]*1e3     
    gks_S[i] = 2*π*ls_S[i]*rs_S[i]*max_gks_S[i]*1e3     
    gca_S[i] = 2*π*ld_S[i]*rd_S[i]*max_gca_S[i]*1e3     

    S_MN = Dict(:Cs => cs_S[i], :Cd => cd_S[i], :Irh => rheobase_S[i], :gc => gc_S[i], :gNa => gna_S[i], :gKs => gks_S[i], :gKf => gkf_S[i], :gCa => gca_S[i], :gld => gld_S[i], :gls => gls_S[i], :αp => αp_D, :βp => βp_D[i], 
        :αq => αq_S[i], :βq => βq_S[i], :αm => αm[i], :βm => βm[i], :αh => αh_S[i], :βh => βh_S[i], :αn => αn_S[i], :βn => βn_S[i], :Vth => Vth_S[i], :VthCa => Vthca_S[i], :Rn => rn_S[i],
        :Tetanic => tetanic_S[i], :Apeak => Apeak_S[i],:IF => IF[i], :tpeak => tpeak_S[i], :γh=>γh[i] , "type" => "All Type")
 
    MN_list = [MN_list;S_MN];
end
MNs = MN_list;


tspan = 0:0.05:sim_length; #time span of simulation defined by step size and length of simulation // change this to get more or less points in the simulation
m=0.0;h=1.0;n=0.0;q=0.0;p=0.0;m0=0.0;h0=1.0;n0=0.0;q0=0.0;p0=0.0; t0=-1.0; t1=-1.0; SPIKE0=0; SPIKE1=0
pool_state_params = fill((m, h, n, q, p, m0, h0, n0, q0, p0, t0, t1, SPIKE0, SPIKE1), nr_s) 
sim_time = _int(sim_length/0.05)   # because the step size is 0.05ms
spike_arr = zeros((nr_s, sim_time)) #storage
vs_arr = zeros((nr_s, sim_time)); #storage
pt_arr = zeros((nr_s, sim_time)); #storage
mt_arr = zeros((nr_s, sim_time)); #storage
ht_arr = zeros((nr_s, sim_time)); #storage
nt_arr = zeros((nr_s, sim_time)); #storage
qt_arr = zeros((nr_s, sim_time)); #storage
commoninput = tr; #needed for the run_sim() function

full_input = zeros((nr_s, sim_length))
independantnoise = zeros((nr_s, sim_length))
factor= 0.1 #for creating the inhibitory signal
inh_input=fill(factor*amplitude, sim_length);



## Optim using libraries

In [ ]:
global fitness_history = Float64[]
global smoothed_fitness_values = []


results = Dict()
for i in 1:nr_s
    println("Checking neuron ", i)

    if hysteresis_exp[i] < 0

        fitness_history = [];best_fitness_val = Inf;iteration = 0 #init

        while best_fitness_val >= 0.005

            println("Starting optimization for Neuron ", i, " Iteration ", iteration)
            
            result=[]
            result = bboptimize(x -> wrap_objective(x, i, Vthca_S, MN_list, hysteresis_exp); SearchRange = [(0., 7.)], Method = :adaptive_de_rand_1_bin_radiuslimited, MaxTime = 200.0, PopulationSize = 100 + 10 * iteration, MutationRate = 0.8 + 0.05 * iteration, CrossoverRate = 0.8 + 0.05 * iteration  )

            best_params = best_candidate(result)
            best_fitness_val = best_fitness(result)
            iteration += 1

            println("Iteration ", iteration, ": Optimized Result for Neuron ", i, ": ", best_params, " with objective value: ", best_fitness_val)
            
            if best_fitness_val < 0.005
                println("Neuron ", i, " optimization succeeded with objective value: ", best_fitness_val)
                results[i] = (minimizer = best_params, minimum = best_fitness_val)
                break

            elseif best_fitness_val > 0.005
                println("Neuron ", i, " optimization did not reach the desired fitness, retrying...")
           
            end

            if iteration > 5
                println("Neuron ", i, " reached the maximum number of iterations.")
                results[i] = (minimizer = best_params, minimum = best_fitness_val)

                break
            end

            results[i] = (minimizer = best_params, minimum = best_fitness_val)

        end

    else
        println("Neuron ", i, " did not meet the conditions")
        results[i] = 0
    end
end


## Save new variables

In [ ]:
VthCa_newVL

In [67]:
######### FOR TA #########

VthCa_newTA= zeros(1,nr_s)

for i in 1:nr_s
    if hysteresis_exp[i]<0
        VthCa_newTA[i]= results[i].minimizer[1]
    else
        VthCa_newTA[i]= 0
    end
end

In [153]:
######### FOR VL #########

VthCa_newVL= zeros(1,nr_s)

for i in 1:nr_s
    if hysteresis_exp[i]<0
        VthCa_newVL[i]= results[i].minimizer[1]
    else
        VthCa_newVL[i]= 0
    end
end

#### Accuracy for the TA

In [ ]:
neuron_indices = Int[]
minimum_values = Float64[]

for (key, value) in results
    if isa(value, NamedTuple) && haskey(value, :minimum)
        push!(neuron_indices, key)
        push!(minimum_values, value.minimum)
    else
        push!(neuron_indices, key)
        push!(minimum_values, NaN) 
    end
end

# Sort by neuron index
sorted_indices = sortperm(neuron_indices)
sorted_neuron_indices = neuron_indices[sorted_indices]
sorted_minimum_values = minimum_values[sorted_indices]

# Plot the data
Plots.plot(sorted_neuron_indices, sorted_minimum_values, xlabel="MN", ylabel="Error", markershape=:circle, markercolor=:blue, line=:solid,grid=false,label=false,ylim=(-0.1, 0.5))
# Plots.savefig("/Users/nmt121/Desktop/figures/errorVL_subject1.pdf")


##### VTHCA distribution 

In [ ]:
Plots.plot([1:nr_s], VthCa_newVL[1,:], xlabel="MN", ylabel="Error", markershape=:circle, markercolor=:blue, line=:solid,grid=false,label=false)
# Plots.savefig("/Users/nmt121/Desktop/figures/VthCA_VLsubject1.pdf")


## SAVE VARIABLES

In [157]:
JLD2.@save "/Users/nmt121/Desktop/Repositories/Motoneuron_Model/Data_MN_Model/Data_final/Subjects_optim_process/VthCa_VL_subject8_gamma1.jld2" VthCa_newVL
JLD2.@save "/Users/nmt121/Desktop/Repositories/Motoneuron_Model/Data_MN_Model/Data_final/Subjects_optim_process/accuracy_optim_VLsubject8_gamma1.jld2" sorted_minimum_values
JLD2.@save "/Users/nmt121/Desktop/Repositories/Motoneuron_Model/Data_MN_Model/Data_final/Subjects_optim_process/indices_optim_VL_subject8_gamma1.jld2" sorted_minimum_values